In [0]:
%pip install databricks-sdk --upgrade

In [0]:
import re
CATALOG = dbutils.widgets.get("CATALOG")

# Coordinates for the shared Lakebase Autoscaling project (created by
# stages/lakebase_project.ipynb).  The refund-manager app reads/writes the
# caspers_refund database inside this project; see stages/lakebase.ipynb
# for the synced table that materialises agent recommendations there.
PROJECT_ID = re.sub(r'[^a-z0-9-]', '-', f"{CATALOG}-caspers".lower())
BRANCH_PATH = f"projects/{PROJECT_ID}/branches/production"
ENDPOINT_PATH = f"{BRANCH_PATH}/endpoints/primary"
LAKEBASE_DB = "caspers-refund"  # Lakebase Autoscale requires DNS-safe names

In [ ]:
import sys
sys.path.append('../utils')
from uc_state import add

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

WAREHOUSE_NAME = f"{CATALOG}-warehouse"
existing_wh = [wh for wh in w.warehouses.list() if wh.name == WAREHOUSE_NAME]
if existing_wh:
    warehouse = existing_wh[0]
    print(f"♻️ Using existing warehouse: {warehouse.id}")
else:
    warehouse = w.warehouses.create(
        name=WAREHOUSE_NAME,
        cluster_size="2X-Small",
        max_num_clusters=1,
        min_num_clusters=1,
        enable_serverless_compute=True,
    ).result()
    add(CATALOG, "warehouses", warehouse)
    print(f"✅ Created warehouse: {warehouse.id}")

In [0]:
print(warehouse)

In [0]:
app_yaml_contents = f"""command:
  - uvicorn
  - app.main:app
env:
  - name: DATABRICKS_WAREHOUSE_ID
    value: '{warehouse.id}'
  - name: DATABRICKS_CATALOG
    value: '{CATALOG}'
  - name: LAKEBASE_ENDPOINT_PATH
    value: '{ENDPOINT_PATH}'
  - name: LAKEBASE_DATABASE_NAME
    value: '{LAKEBASE_DB}'
"""

# Rewriting app.yaml on every stage run gives downstream apps deterministic
# env vars regardless of how many bundle deploys have happened.
import os, time
if os.path.exists("../apps/refund-manager/app.yaml"):
    os.remove("../apps/refund-manager/app.yaml")
time.sleep(3)

with open("../apps/refund-manager/app.yaml", "w") as f:
    f.write(app_yaml_contents)

In [0]:
from databricks.sdk.service.apps import App, AppResource, AppResourceSqlWarehouse, AppResourceSqlWarehouseSqlWarehousePermission
import os
import re as _re

source_code_path = os.path.abspath("../apps/refund-manager")

# P1-18: catalog-scope the app name so two users on the same workspace don't
# fight over the global `refundmanager` slug.  Sanitise to lowercase
# alphanumerics+hyphens (Databricks Apps constraint) and cap at 30 chars.
APP_NAME = _re.sub(r"-+", "-", _re.sub(r"[^a-z0-9-]", "-", f"refundmanager-{CATALOG}".lower())).strip("-")[:30]
print(f"App name: {APP_NAME}")

# AppResourceDatabase is the Provisioned-Lakebase binding; the consolidated
# Autoscaling project doesn't need (and can't accept) it.  The app reaches
# Lakebase via its own service-principal OAuth + a per-endpoint credential
# minted in app/db.py.  See the role-grant cell below for the Postgres-side
# binding that gives the app's SP a role in caspers_refund.
APP_DESCRIPTION = (
    "Casper's Refund Manager — operator console for the refund recommender "
    "agent: queue of pending orders, structured agent decisions, and "
    "one-click application of refunds against Lakebase."
)
app_def = App(
    name=APP_NAME,
    description=APP_DESCRIPTION,
    default_source_code_path=source_code_path,
    resources=[
        AppResource(
            name="sql-warehouse",
            sql_warehouse=AppResourceSqlWarehouse(
                id=warehouse.id,
                permission=AppResourceSqlWarehouseSqlWarehousePermission.CAN_USE),
        ),
    ],
)

try:
    existing_app = w.apps.get(APP_NAME)
    print(f"♻️ App {APP_NAME} already exists, updating...")
    app = w.apps.update(APP_NAME, app_def)
except Exception:
    app = w.apps.create(app_def)

In [0]:
import time

def _app_state(a):
    cs = getattr(a, "compute_status", None)
    s = getattr(cs, "state", None) if cs is not None else None
    if s is None:
        s = getattr(a, "state", None)
    return getattr(s, "value", str(s)) if s is not None else ""

deadline = time.time() + 30 * 60
while True:
    current = w.apps.get(APP_NAME)
    state = _app_state(current)
    print(f"App {APP_NAME} state: {state}")
    if state in ("ACTIVE", "RUNNING", "READY"):
        app_status = current
        break
    if state in ("ERROR", "FAILED"):
        raise RuntimeError(f"App {APP_NAME} entered failure state: {state}")
    if time.time() > deadline:
        raise TimeoutError(f"App {APP_NAME} not ready after 30 minutes (last state: {state})")
    time.sleep(15)

add(CATALOG, "apps", app_status)

# Discover Domain tagging.  The refund-manager app surfaces in BOTH the
# Operations and Revenue & Customers domains: ops/manager personas use it
# to reverse-decide automated refunds (operational tool), and the same
# data drives revenue analytics (refund leakage, cohort impact).  See
# SETUP.ipynb §4 for the manual one-time Domain creation step.
import sys as _sys, os as _os
_sys.path.insert(0, _os.path.abspath(".."))  # stages/ -> repo root
from utils.domain_tags import ensure_domain_tag_policies, tag_workspace_entity
print("\n— Tagging refund-manager app for Discover Domains —")
ensure_domain_tag_policies(w, verbose=False)
tag_workspace_entity(w, "apps", APP_NAME, ["operations", "revenue"],
                     label=f"app {APP_NAME!r}")

In [ ]:
# Upload the app thumbnail.  Best-effort: purely visual polish for the
# Discover / Workspace card and the app launcher icon, so we never let a
# missing API surface fail the whole stage.
#
# Runtime requirement: databricks-sdk >= 0.83 (has w.apps.update_app_thumbnail).
# Thumbnail source: utils/casperslogo_green.png (synced by DABs).
import base64, os

try:
    from databricks.sdk.service.apps import AppThumbnail

    if not hasattr(w.apps, "update_app_thumbnail"):
        print("\u26a0\ufe0f  w.apps.update_app_thumbnail not available "
              "(databricks-sdk too old) \u2014 skipping thumbnail upload.")
    else:
        thumb_path = os.path.abspath("../utils/casperslogo_green.png")
        with open(thumb_path, "rb") as _f:
            thumb_b64 = base64.b64encode(_f.read()).decode("ascii")
        w.apps.update_app_thumbnail(
            APP_NAME,
            app_thumbnail=AppThumbnail(thumbnail=thumb_b64),
        )
        print(f"\U0001f5bc\ufe0f  Uploaded thumbnail for {APP_NAME}")
except Exception as e:
    print(f"\u26a0\ufe0f  Could not upload thumbnail for {APP_NAME}: {e}")

In [ ]:
# Grant app permissions to UC.  The refund manager app calls the refund
# agent endpoint (and, via the supervisor flow surfaced in the ops app,
# may transitively trigger other agents) which is deployed in OBO mode —
# the app SP's identity propagates all the way down to the UC function
# calls inside the agent.  Without USE_SCHEMA + EXECUTE on `ai`, the
# agent surfaces PermissionError("EXECUTE on Routine '<catalog>.ai.<fn>'")
# on the first tool call.  The `account users` grant the refunder_agent
# stage applies does NOT cover app SPs — workspace-local app SPs are not
# members of `account users` (an account-level group).
from databricks.sdk.service import catalog

for full_name, securable_type, privilege in [
    (f"{CATALOG}",                       "CATALOG", catalog.Privilege.USE_CATALOG),
    (f"{CATALOG}.ai",                    "SCHEMA",  catalog.Privilege.USE_SCHEMA),
    (f"{CATALOG}.ai",                    "SCHEMA",  catalog.Privilege.EXECUTE),
    (f"{CATALOG}.lakeflow",              "SCHEMA",  catalog.Privilege.USE_SCHEMA),
    (f"{CATALOG}.lakeflow.all_events",   "TABLE",   catalog.Privilege.SELECT),
    (f"{CATALOG}.simulator",             "SCHEMA",  catalog.Privilege.USE_SCHEMA),
    (f"{CATALOG}.simulator.locations",   "TABLE",   catalog.Privilege.SELECT),
]:
    try:
        w.grants.update(
            full_name=full_name,
            securable_type=securable_type,
            changes=[
                catalog.PermissionsChange(
                    add=[privilege],
                    principal=app_status.id,
                )
            ],
        )
        print(f"\u2705 Granted {privilege} on {securable_type} {full_name}")
    except Exception as e:
        print(f"\u26a0\ufe0f  Could not grant {privilege} on {full_name}: {e}")

In [0]:
from databricks.sdk.common.types.fieldmask import FieldMask

from databricks.sdk.service.postgres import (
      Role,
      RoleRoleSpec,
      RoleMembershipRole,
      RoleIdentityType,
)

# Shared Autoscale project provisioned by stages/lakebase_project.ipynb.
PROJECT = f"projects/{PROJECT_ID}"

# Pick the production branch by `status.default` rather than list index;
# the Postgres API doesn't guarantee list ordering.
branches = list(w.postgres.list_branches(PROJECT))
PRODUCTION = next(
    (b for b in branches if getattr(getattr(b, "status", None), "default", False)),
    branches[0],
).name

# Resolve the app's service-principal client ID.  The app's pod authenticates
# to Lakebase as this SP via OAuth, so we need a Postgres role bound to this
# identity in order to GRANT it access to the caspers_refund DB.
app_sp_id = (
    getattr(app_status, "service_principal_client_id", None)
    or (app_status.as_dict() if hasattr(app_status, "as_dict") else {}).get("service_principal_client_id")
)
assert app_sp_id, "Could not determine refund-manager app service principal ID"

# Find-or-create-or-update the SP's role at DATABRICKS_SUPERUSER.  This
# mirrors stages/operational_app.ipynb's pattern for the same project; both
# apps need the same membership in the shared Autoscale project.  The role
# is scoped to the branch (production), not to a specific database — the
# SUPERUSER membership covers every DB in the project, which is fine for a
# demo (in prod you'd grant per-database privileges instead).
try:
    existing_roles = list(w.postgres.list_roles(PRODUCTION))
    app_role = next(
        (r for r in existing_roles
         if getattr(getattr(r, "spec", None), "postgres_role", None) == app_sp_id),
        None,
    )
    if app_role:
        app_role.spec = RoleRoleSpec(
            identity_type=RoleIdentityType.SERVICE_PRINCIPAL,
            postgres_role=app_sp_id,
            membership_roles=[RoleMembershipRole.DATABRICKS_SUPERUSER],
        )
        w.postgres.update_role(
            name=app_role.name,
            role=app_role,
            update_mask=FieldMask(field_mask=["spec.membership_roles"]),
        )
        print(f"✅ Updated app role to DATABRICKS_SUPERUSER (SP {app_sp_id})")
    else:
        try:
            w.postgres.create_role(
                parent=PRODUCTION,
                role=Role(
                    spec=RoleRoleSpec(
                        identity_type=RoleIdentityType.SERVICE_PRINCIPAL,
                        postgres_role=app_sp_id,
                        membership_roles=[RoleMembershipRole.DATABRICKS_SUPERUSER],
                    ),
                ),
            )
            print(f"✅ Created DATABRICKS_SUPERUSER role for app SP {app_sp_id}")
        except Exception as _ce:
            # Idempotent path: if the server says the role already exists,
            # the list_roles() match above didn't recognise it (most likely
            # because spec.postgres_role on the returned role isn't exactly
            # equal to the SP client_id we used to filter — UUID casing or
            # a different identity field).  The role is already configured
            # for our SP, so treat as success and continue.
            _msg = str(_ce).lower()
            if "already exists" in _msg:
                print(f"♻ DATABRICKS_SUPERUSER role for app SP {app_sp_id} already exists — skipping create")
            else:
                raise
except Exception as e:
    # Non-fatal: the app deploy below still proceeds.  If the SP grant didn't
    # land, the app's first Postgres query will surface the error and we can
    # repair manually.
    print(f"⚠\ufe0f Could not grant Lakebase role to app SP: {type(e).__name__}: {e}")

In [0]:
app_status

In [0]:
import time
from databricks.sdk.service.apps import AppDeployment

deployment = w.apps.deploy(
    app_name=app_status.name,
    app_deployment=AppDeployment(
        source_code_path=source_code_path
    )
)

def _deploy_state(d):
    st = getattr(d, "status", None)
    s = getattr(st, "state", None) if st is not None else None
    return getattr(s, "value", str(s)) if s is not None else ""

deadline = time.time() + 30 * 60
while True:
    current_dep = w.apps.get_deployment(app_name=app_status.name, deployment_id=deployment.deployment_id)
    state = _deploy_state(current_dep)
    print(f"Deployment state: {state}")
    if state == "SUCCEEDED":
        deployment_status = current_dep
        break
    if state in ("FAILED", "STOPPED"):
        raise RuntimeError(f"Deployment failed for {app_status.name}: state={state}")
    if time.time() > deadline:
        raise TimeoutError(f"Deployment for {app_status.name} not ready after 30 minutes (last state: {state})")
    time.sleep(10)

display(deployment_status)